# Phase 2b: Entity分布分析

**目标**：分析626个entities在14个标准relations中的分布，为Entity聚类提供数据支持

**输入**：
- `results/phase1_5percent_exploration.json` - Phase 1原始数据
- `results/relation_mapping_final_v1.json` - Relation映射

**输出**：
- 每个relation的entity统计
- 频率分布可视化
- 聚类策略建议

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from pathlib import Path

# 设置绘图样式
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
sns.set_palette('husl')

## 1. 加载数据

In [ ]:
# 加载Phase 1原始数据
with open('../results/phase1_5percent_exploration.json', 'r') as f:
    phase1_data_full = json.load(f)

# 提取results列表
phase1_results = phase1_data_full['results']

print(f"Phase 1数据: {len(phase1_results)} 部电影")
print(f"\n数据格式示例:")
print(f"  RecBole ID: {phase1_results[0]['recbole_id']}")
print(f"  Original Movie ID: {phase1_results[0]['original_movie_id']}")
print(f"  知识点数: {phase1_results[0]['num_knowledge_points']}")
print(f"  Status: {phase1_results[0]['status']}")

In [ ]:
# 加载Relation映射
with open('../results/relation_mapping_final_v1.json', 'r') as f:
    relation_mapping_data = json.load(f)

relation_mapping = relation_mapping_data['relation_mapping']
standard_relations = relation_mapping_data['standard_relations']

print(f"14个标准Relations:")
for i, rel in enumerate(standard_relations, 1):
    print(f"{i:2d}. {rel}")

## 2. 提取所有知识点并映射到标准relations

In [ ]:
# 提取所有知识点（只处理成功提取的电影）
all_knowledge_points = []

for movie_result in phase1_results:
    if movie_result['status'] != 'success':
        continue
    
    recbole_id = movie_result['recbole_id']
    
    for kp in movie_result['knowledge_points']:
        # 映射到标准relation
        original_relation = kp['relation']
        standard_relation = relation_mapping.get(original_relation, 'others_relation')
        
        all_knowledge_points.append({
            'recbole_id': recbole_id,
            'original_relation': original_relation,
            'standard_relation': standard_relation,
            'entity': kp['entity']
        })

print(f"总知识点数: {len(all_knowledge_points)}")
print(f"\n前5个知识点示例:")
for i, kp in enumerate(all_knowledge_points[:5], 1):
    print(f"{i}. {kp['standard_relation']:20s} -> {kp['entity']}")

## 3. 统计每个标准relation的entity分布

In [ ]:
# 按标准relation分组统计entities
relation_entities = defaultdict(list)

for kp in all_knowledge_points:
    relation_entities[kp['standard_relation']].append(kp['entity'])

# 统计每个relation的详细信息
entity_stats = {}

for relation, entities in relation_entities.items():
    entity_counter = Counter(entities)
    unique_entities = len(entity_counter)
    total_instances = len(entities)
    
    # 频率分布统计
    freq_1 = sum(1 for count in entity_counter.values() if count == 1)
    freq_2_4 = sum(1 for count in entity_counter.values() if 2 <= count <= 4)
    freq_5_10 = sum(1 for count in entity_counter.values() if 5 <= count <= 10)
    freq_10_plus = sum(1 for count in entity_counter.values() if count > 10)
    
    entity_stats[relation] = {
        'unique_entities': unique_entities,
        'total_instances': total_instances,
        'entity_counter': entity_counter,
        'freq_distribution': {
            'freq_1': freq_1,
            'freq_2_4': freq_2_4,
            'freq_5_10': freq_5_10,
            'freq_10+': freq_10_plus
        },
        'top_entities': entity_counter.most_common(10)
    }

print("每个Relation的Entity统计:")
print("="*100)

# 按unique_entities排序显示
sorted_relations = sorted(entity_stats.items(), key=lambda x: x[1]['unique_entities'], reverse=True)

for relation, stats in sorted_relations:
    print(f"\n{relation}:")
    print(f"  Unique entities: {stats['unique_entities']:3d}")
    print(f"  Total instances: {stats['total_instances']:3d}")
    print(f"  频率分布: freq=1: {stats['freq_distribution']['freq_1']:3d}, "
          f"freq=2-4: {stats['freq_distribution']['freq_2_4']:3d}, "
          f"freq=5-10: {stats['freq_distribution']['freq_5_10']:3d}, "
          f"freq>10: {stats['freq_distribution']['freq_10+']:3d}")
    print(f"  Top 5 entities:")
    for entity, count in stats['top_entities'][:5]:
        print(f"    - {entity:40s}: {count:3d}")

## 4. 全局统计

In [ ]:
# 全局统计
total_unique_entities = sum(stats['unique_entities'] for stats in entity_stats.values())
total_instances = sum(stats['total_instances'] for stats in entity_stats.values())

# 统计所有entity的频率（跨relation统计全局唯一entity数）
all_entities = [kp['entity'] for kp in all_knowledge_points]
all_entity_counter = Counter(all_entities)
global_unique = len(all_entity_counter)

global_freq_1 = sum(1 for count in all_entity_counter.values() if count == 1)
global_freq_2_4 = sum(1 for count in all_entity_counter.values() if 2 <= count <= 4)
global_freq_5_plus = sum(1 for count in all_entity_counter.values() if count >= 5)

print("="*100)
print("全局统计")
print("="*100)
print(f"14个标准relations")
print(f"总unique entities (按relation累加): {total_unique_entities}")
print(f"全局unique entities (跨relation去重): {global_unique}")
print(f"总instances: {total_instances}")
print(f"平均每个relation: {total_unique_entities / 14:.1f} unique entities")
print(f"\n全局频率分布 (跨relation统计):")
print(f"  只出现1次的entities: {global_freq_1} ({global_freq_1/global_unique*100:.1f}%)")
print(f"  出现2-4次的entities: {global_freq_2_4} ({global_freq_2_4/global_unique*100:.1f}%)")
print(f"  出现5次以上的entities: {global_freq_5_plus} ({global_freq_5_plus/global_unique*100:.1f}%)")
print(f"\nEntity复用率 (全局): {total_instances / global_unique:.2f}x")

## 5. 可视化

In [ ]:
# 图表A: 每个relation的entity数量（横向条形图）
fig, ax = plt.subplots(figsize=(14, 10))

relations_list = [r for r, _ in sorted_relations]
unique_counts = [entity_stats[r]['unique_entities'] for r in relations_list]
total_counts = [entity_stats[r]['total_instances'] for r in relations_list]

y_pos = np.arange(len(relations_list))

# 绘制两个柱状图
bars1 = ax.barh(y_pos + 0.2, unique_counts, 0.4, label='Unique entities', color='steelblue')
bars2 = ax.barh(y_pos - 0.2, total_counts, 0.4, label='Total instances', color='coral', alpha=0.7)

ax.set_yticks(y_pos)
ax.set_yticklabels(relations_list)
ax.invert_yaxis()
ax.set_xlabel('Count')
ax.set_title('Entity数量分布（按Relation）', fontsize=16, fontweight='bold')
ax.legend()

# 在柱子上标注数字
for i, (bar1, bar2) in enumerate(zip(bars1, bars2)):
    ax.text(bar1.get_width() + 2, bar1.get_y() + bar1.get_height()/2, 
            f'{int(bar1.get_width())}', va='center', fontsize=9)
    ax.text(bar2.get_width() + 2, bar2.get_y() + bar2.get_height()/2, 
            f'{int(bar2.get_width())}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../results/entity_distribution_by_relation.png', dpi=150, bbox_inches='tight')
plt.show()

print("图表已保存: results/entity_distribution_by_relation.png")

In [ ]:
# 图表B: 频率分布堆叠柱状图
fig, ax = plt.subplots(figsize=(16, 8))

relations_list = [r for r, _ in sorted_relations]
freq_1_list = [entity_stats[r]['freq_distribution']['freq_1'] for r in relations_list]
freq_2_4_list = [entity_stats[r]['freq_distribution']['freq_2_4'] for r in relations_list]
freq_5_10_list = [entity_stats[r]['freq_distribution']['freq_5_10'] for r in relations_list]
freq_10_plus_list = [entity_stats[r]['freq_distribution']['freq_10+'] for r in relations_list]

x = np.arange(len(relations_list))
width = 0.6

# 堆叠柱状图
p1 = ax.bar(x, freq_1_list, width, label='freq=1 (singleton)', color='#d62728')
p2 = ax.bar(x, freq_2_4_list, width, bottom=freq_1_list, label='freq=2-4', color='#ff7f0e')
p3 = ax.bar(x, freq_5_10_list, width, 
           bottom=np.array(freq_1_list) + np.array(freq_2_4_list), 
           label='freq=5-10', color='#2ca02c')
p4 = ax.bar(x, freq_10_plus_list, width, 
           bottom=np.array(freq_1_list) + np.array(freq_2_4_list) + np.array(freq_5_10_list), 
           label='freq>10', color='#1f77b4')

ax.set_xlabel('Relation')
ax.set_ylabel('Entity数量')
ax.set_title('Entity频率分布（堆叠图）', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(relations_list, rotation=45, ha='right')
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/entity_frequency_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("图表已保存: results/entity_frequency_distribution.png")

## 6. 聚类难度评估与策略建议

In [ ]:
# 根据entity数量分类
category_large = []   # > 60 entities
category_medium = []  # 20-60 entities
category_small = []   # < 20 entities

for relation, stats in entity_stats.items():
    unique = stats['unique_entities']
    if unique > 60:
        category_large.append((relation, unique, stats['total_instances']))
    elif unique >= 20:
        category_medium.append((relation, unique, stats['total_instances']))
    else:
        category_small.append((relation, unique, stats['total_instances']))

print("="*100)
print("聚类难度评估与策略建议")
print("="*100)

print(f"\n【需要重点聚类】(unique entities > 60):")
for rel, unique, total in sorted(category_large, key=lambda x: x[1], reverse=True):
    print(f"  - {rel:25s}: {unique:3d} entities ({total:3d} instances)")
    print(f"    建议: 聚类到 25-35 个标准entities")

print(f"\n【适度聚类】(20 ≤ unique entities ≤ 60):")
for rel, unique, total in sorted(category_medium, key=lambda x: x[1], reverse=True):
    print(f"  - {rel:25s}: {unique:3d} entities ({total:3d} instances)")
    target = int(unique * 0.5)
    print(f"    建议: 聚类到 {target} 个标准entities (压缩50%)")

print(f"\n【轻量处理】(unique entities < 20):")
for rel, unique, total in sorted(category_small, key=lambda x: x[1], reverse=True):
    freq_1 = entity_stats[rel]['freq_distribution']['freq_1']
    print(f"  - {rel:25s}: {unique:3d} entities ({total:3d} instances, {freq_1} singletons)")
    if freq_1 > unique * 0.5:
        print(f"    建议: 保留高频entities，合并singletons到others_entity")
    else:
        print(f"    建议: 直接保留所有，或轻量合并")

## 7. 计算聚类目标数量

In [ ]:
# 自适应聚类策略
clustering_targets = {}

for relation, stats in entity_stats.items():
    unique = stats['unique_entities']
    freq_1 = stats['freq_distribution']['freq_1']
    
    # 策略规则
    if unique < 10:
        # 极少entities，保留全部或只合并freq=1
        target = unique - freq_1 + 1  # 所有freq>1的 + 1个others
        strategy = 'preserve_high_freq'
    elif 10 <= unique < 30:
        # 小规模，轻量聚类
        target = max(int(unique * 0.6), 8)  # 压缩到60%，最少8个
        strategy = 'light_clustering'
    elif 30 <= unique < 80:
        # 中等规模，适度聚类
        target = 20
        strategy = 'moderate_clustering'
    else:  # >= 80
        # 大规模，重度聚类
        target = 30
        strategy = 'heavy_clustering'
    
    clustering_targets[relation] = {
        'current': unique,
        'target': target,
        'strategy': strategy,
        'compression_rate': target / unique if unique > 0 else 1.0
    }

print("="*100)
print("聚类目标规划")
print("="*100)
print(f"\n{'Relation':25s} {'当前':>6s} {'目标':>6s} {'压缩率':>8s} {'策略':20s}")
print("-"*100)

total_current = 0
total_target = 0

for relation in sorted(clustering_targets.keys()):
    info = clustering_targets[relation]
    total_current += info['current']
    total_target += info['target']
    
    print(f"{relation:25s} {info['current']:6d} {info['target']:6d} "
          f"{info['compression_rate']:7.1%} {info['strategy']:20s}")

print("-"*100)
print(f"{'总计':25s} {total_current:6d} {total_target:6d} {total_target/total_current:7.1%}")
print(f"\n全局压缩率: {total_current} → {total_target} ({total_target/total_current*100:.1f}%)")

## 8. 保存统计结果

In [ ]:
# 保存统计数据供后续使用
output_data = {
    'metadata': {
        'total_knowledge_points': len(all_knowledge_points),
        'total_unique_entities_by_relation': total_unique_entities,
        'global_unique_entities': global_unique,
        'num_standard_relations': len(standard_relations),
        'analysis_date': '2026-01-02'
    },
    'entity_stats_by_relation': {
        relation: {
            'unique_entities': stats['unique_entities'],
            'total_instances': stats['total_instances'],
            'freq_distribution': stats['freq_distribution'],
            'top_10_entities': dict(stats['top_entities'])
        }
        for relation, stats in entity_stats.items()
    },
    'clustering_targets': clustering_targets,
    'summary': {
        'current_total': total_current,
        'target_total': total_target,
        'compression_rate': total_target / total_current
    }
}

with open('../results/entity_distribution_analysis.json', 'w') as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)

print("统计数据已保存: results/entity_distribution_analysis.json")

## 9. 总结

**数据探索完成！**

**关键发现**：
- 全局unique entities (跨relation去重): 见上方统计
- 每个relation平均XX个entities
- XX%的entities只出现1次（长尾分布）

**聚类策略**：
- 自适应策略：根据每个relation的entity数量调整聚类目标
- 预计压缩到XX个标准entities

**下一步**：
1. 基于这个分析结果，实现EntityClusterer类
2. 对每个relation分别聚类
3. 生成完整的知识词典v1

## 10. 轻量级Relations详细展示（用于手动聚类）

展示以下6个轻量级relation的所有entities及其频数：
- color_palette
- interaction
- graphic_element
- genre
- symbolism
- others_relation

In [ ]:
# 定义轻量级relations
LIGHT_RELATIONS = [
    'color_palette',
    'interaction',
    'graphic_element',
    'genre',
    'symbolism',
    'others_relation'
]

print("="*120)
print("轻量级Relations详细Entity列表（含原始Relation溯源）")
print("="*120)

# 为每个轻量relation，构建entity -> 原始relations的映射
for relation in LIGHT_RELATIONS:
    if relation not in entity_stats:
        print(f"\n{relation}: 无数据")
        continue

    # 收集该标准relation下的所有knowledge points
    entity_to_original_relations = defaultdict(lambda: defaultdict(int))  # {entity: {original_relation: count}}

    for kp in all_knowledge_points:
        if kp['standard_relation'] == relation:
            entity = kp['entity']
            original_relation = kp['original_relation']
            entity_to_original_relations[entity][original_relation] += 1

    # 计算总频数
    entity_total_counts = {
        entity: sum(orig_rels.values())
        for entity, orig_rels in entity_to_original_relations.items()
    }

    # 按总频数降序排序
    sorted_entities = sorted(entity_total_counts.items(), key=lambda x: (-x[1], x[0]))

    stats = entity_stats[relation]

    print(f"\n{'='*120}")
    print(f"📋 {relation.upper()}")
    print(f"{'='*120}")
    print(f"总unique entities: {stats['unique_entities']}")
    print(f"总instances: {stats['total_instances']}")
    print(f"Singletons (freq=1): {stats['freq_distribution']['freq_1']}")
    print()
    print(f"{'序号':>4}  {'Entity':<45}  {'频数':>6}  {'原始Relations (频数)':}")
    print("-"*120)

    for idx, (entity, total_count) in enumerate(sorted_entities, 1):
        original_relations = entity_to_original_relations[entity]

        # 格式化原始relations信息
        orig_rel_strs = [f"{orig_rel}({count})" for orig_rel, count in
                         sorted(original_relations.items(), key=lambda x: -x[1])]
        orig_rel_display = ", ".join(orig_rel_strs)

        print(f"{idx:4d}  {entity:<45}  {total_count:6d}  {orig_rel_display}")

    print()